# Case 05 — Regularization cho LogReg: **L1 / L2 / ElasticNet**  ·  *Bai L5*

## Y tuong
Regularization = them "hinh phat" len do lon he so de mo hinh khong qua khop va on dinh hon:

| Loai | Cach phat | Tac dung dac trung |
|---|---|---|
| **L2 (Ridge)** | binh phuong he so | co nho DEU moi he so — hop khi co cong tuyen (Case 04) |
| **L1 (Lasso)** | tri tuyet doi he so | **EP han mot so he so ve 0** -> tu loai feature (Embedded) |
| **ElasticNet** | tron L1 + L2 | vua co vua loai |

## Vi sao chay thu nghiem nay
Notebook chinh chi khai bao `penalty='l2'`, bo phi 2 lua chon kia du DA HOC. Ta so ca ba tren cung du lieu.

In [1]:
# ============================================================================
# PRELUDE DUNG CHUNG cho moi notebook trong thu muc nay.
# Muc dich: moi file TU CHAY DUOC ma khong phu thuoc notebook chinh.
#   - Nap train.csv (Day chuyen A) + test.csv (Day chuyen B)
#   - Tao lai dung 4 feature co che nhu notebook chinh (de ket qua nhat quan)
# ============================================================================
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)   # co dinh seed -> chay lai ra so giong nhau

# --- Nap du lieu: thu nhieu duong dan vi notebook nam trong thu muc con ---
CANDIDATES = ['../Data_Final/Data_Final','../Data_Final','Data_Final/Data_Final',
              '../../Data_Final/Data_Final','Data_Final','.']
DATA_DIR = next((c for c in CANDIDATES
                 if os.path.exists(os.path.join(c,'train.csv'))
                 and os.path.exists(os.path.join(c,'test.csv'))), None)
assert DATA_DIR is not None, 'Khong tim thay train.csv/test.csv'
train = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))

NUM = ['nhiet_do_moi_truong','nhiet_do_quy_trinh','toc_do_quay','momen_xoan','do_mon_dao']  # 5 bien so goc
CAT = ['loai_san_pham','ca_lam_viec']   # 2 bien phan loai goc
TARGET = 'hong_hoc'                      # nhan: 1 = hong trong ca ke tiep

# --- Feature Engineering theo CO CHE vat ly (giong het notebook chinh) ---
def add_features(df):
    d = df.copy()
    # (1) Chenh lech nhiet: co che tan nhiet kem HDF. Lay HIEU nen triet tieu offset khi hau -> giam shift.
    d['chenh_lech_nhiet'] = d['nhiet_do_quy_trinh'] - d['nhiet_do_moi_truong']
    # (2) Cong suat co (W) = momen(Nm) * van_toc_goc(rad/s); van_toc_goc = rpm*2pi/60. Co che qua tai cong suat PWF.
    d['cong_suat_co']     = d['momen_xoan'] * d['toc_do_quay'] * 2*np.pi/60.0
    # (3) Tich mon*momen: co che qua tai cang thang OSF (mon cang nhieu + momen cang lon -> cang de gay).
    d['tich_mon_momen']   = d['do_mon_dao'] * d['momen_xoan']
    # (4) osf_margin: khoang cach toi NGUONG OSF, nguong phu thuoc HANG SP (L/M/H). >0 = da vuot nguong.
    g = d['loai_san_pham'].map({'L':11000,'M':12000,'H':13000})
    d['osf_margin']       = d['tich_mon_momen'] - g
    return d

train_fe = add_features(train); test_fe = add_features(test)
y_train = train_fe[TARGET].values
y_test  = test_fe[TARGET].values

# Bo feature CUOI da chot o notebook chinh: 9 bien so + 1 bien phan loai (loai_san_pham)
FINAL_NUM = NUM + ['chenh_lech_nhiet','cong_suat_co','tich_mon_momen','osf_margin']
FINAL_CAT = ['loai_san_pham']
print('Train:', train.shape, '| Test:', test.shape,
      '| ti le hong train/test:', round(y_train.mean(),3), round(y_test.mean(),3))
print('FINAL_NUM =', FINAL_NUM)

Train: (14000, 8) | Test: (6000, 8) | ti le hong train/test: 0.074 0.08
FINAL_NUM = ['nhiet_do_moi_truong', 'nhiet_do_quy_trinh', 'toc_do_quay', 'momen_xoan', 'do_mon_dao', 'chenh_lech_nhiet', 'cong_suat_co', 'tich_mon_momen', 'osf_margin']


In [2]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, average_precision_score
from scipy.stats import loguniform
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

pre = ColumnTransformer([('num', StandardScaler(), FINAL_NUM),
      ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), FINAL_CAT)])

def run(penalty, extra):
    # solver='saga' la solver duy nhat ho tro CA l1, l2 va elasticnet
    lr   = LogisticRegression(solver='saga', penalty=penalty, max_iter=5000, class_weight='balanced')
    grid = {'lr__C': loguniform(1e-2, 1e2)}; grid.update(extra)   # C = 1/lambda (C nho = phat manh)
    pipe = Pipeline([('pre', pre), ('lr', lr)])
    rs   = RandomizedSearchCV(pipe, grid, n_iter=12, scoring='average_precision',
                              cv=skf, random_state=RANDOM_STATE, n_jobs=-1
                              ).fit(train_fe[FINAL_NUM+FINAL_CAT], y_train)
    best = rs.best_estimator_
    # nguong F1 lay tu CV tren train (khong nhin test)
    oof = cross_val_predict(best, train_fe[FINAL_NUM+FINAL_CAT], y_train, cv=skf, method='predict_proba', n_jobs=-1)[:,1]
    ts  = np.linspace(0.05,0.9,80); thr = ts[int(np.argmax([f1_score(y_train,(oof>=t)) for t in ts]))]
    p    = best.predict_proba(test_fe[FINAL_NUM+FINAL_CAT])[:,1]
    coef = best.named_steps['lr'].coef_[0]
    return {'penalty':penalty, 'best_C':round(rs.best_params_['lr__C'],3),
            'he_so_khac_0':int((np.abs(coef)>1e-6).sum()), 'tong_he_so':len(coef),
            'AUC_PR':average_precision_score(y_test,p), 'F1':f1_score(y_test,(p>=thr))}

rows = [run('l2', {}),
        run('l1', {}),
        run('elasticnet', {'lr__l1_ratio':[0.2, 0.5, 0.8]})]   # l1_ratio = ty le tron L1 trong ElasticNet
display(pd.DataFrame(rows).set_index('penalty').round(4))

,best_C,he_so_khac_0,tong_he_so,AUC_PR,F1
penalty,,,,,
l2,0.042,12,12,0.2347,0.3025
l1,0.042,9,12,0.2381,0.3061
elasticnet,0.042,9,12,0.2374,0.3083


In [3]:
# L1 loai feature nao? Xem he so nao bi ep ve 0.
best_l1 = LogisticRegression(solver='saga', penalty='l1', C=0.1, max_iter=5000, class_weight='balanced')
pipe_l1 = Pipeline([('pre', pre), ('lr', best_l1)]).fit(train_fe[FINAL_NUM+FINAL_CAT], y_train)
names = pipe_l1.named_steps['pre'].get_feature_names_out()
coef  = pipe_l1.named_steps['lr'].coef_[0]
imp   = pd.Series(coef, index=names).sort_values(key=np.abs, ascending=False)
print('He so LogReg-L1 (C=0.1), sap theo do lon:'); print(imp.round(3).to_string())
print('\n=> feature co he so = 0 la feature bi L1 LOAI BO (tu chon feature kieu Embedded).')

He so LogReg-L1 (C=0.1), sap theo do lon:
num__momen_xoan            -2.445
num__cong_suat_co           2.299
num__toc_do_quay           -1.164
num__tich_mon_momen         0.674
num__chenh_lech_nhiet      -0.314
num__do_mon_dao             0.123
cat__loai_san_pham_M       -0.115
num__osf_margin             0.020
num__nhiet_do_moi_truong    0.010
num__nhiet_do_quy_trinh     0.000
cat__loai_san_pham_H        0.000
cat__loai_san_pham_L        0.000

=> feature co he so = 0 la feature bi L1 LOAI BO (tu chon feature kieu Embedded).


> ### 🔎 Doc ket qua (Case 05)
> | Penalty | He so khac 0 | AUC-PR | F1 |
> |---|---|---|---|
> | L2 | 12/12 | 0.235 | 0.302 |
> | L1 | 9/12 | 0.238 | **0.306** |
> | ElasticNet | 9/12 | 0.237 | **0.308** |
>
> - **L1 & ElasticNet nhinh hon L2** mot chut ve F1.
> - **L1 ep 3 he so ve dung 0** (`nhiet_do_quy_trinh` va 2 muc cua `loai_san_pham`) -> tu noi
>   "3 feature nay bo cung duoc", **khop voi ket luan dong thuan o Case 03**.
>
> ### ✅ Ket luan Case 05
> Giu `penalty='l2'` la hop ly (on dinh khi co cong tuyen — Case 04), nhung nen **bo sung `penalty` vao grid**
> cua RandomizedSearchCV (`l1`/`l2`/`elasticnet`, solver `saga`) de may tu chon — dung tinh than L5, va cho
> cau chuyen "regularization vua chong overfit vua on dinh hoa cong tuyen".